In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [4]:
# The tutorial uses pandas to read the CSV file and use the information from the dataframe to come up with a schema

import pandas as pd

In [ ]:
# This throws an error. However, it is not  required as the schema is defined for us below

# df_green_pd = pd.read_csv('data/raw/green/2020/01/green_tripdata_2020_01.csv.gz', nrows=100)

In [ ]:
# Note this also creates an error. Not really required

# spark.createDataFrame(df_green_pd.pd)

In [3]:
from pyspark.sql import types

In [26]:
green_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("lpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("lpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("RatecodeID", types.IntegerType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("passenger_count", types.IntegerType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("ehail_fee", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("trip_type", types.IntegerType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

yellow_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.IntegerType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.IntegerType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

In [ ]:
# Reads the CSV file from the input_path, applying the green_schema and writes the transformed data to Parquet format in output_path
# Note: this works. It only throws an error on 2021/08 because of empty data. 
years = [2020,2021]

for year in years:
    for month in range(1, 13):
        print(f'processing data for {year}/{month}')
    
        input_path = f'data/raw/green/{year}/{month:02d}/'
        output_path = f'data/pq/green/{year}/{month:02d}/'
    
        df_green = spark.read \
            .option("header", "true") \
            .schema(green_schema) \
            .csv(input_path)
    
        df_green.repartition(4).write.parquet(output_path)

In [ ]:
#Yellow
# Note: this works. It only throws an error on 2021/08 because of empty data. 
# Note: Not sure the repartitions worked. To be checked.

years = [2020,2021]

for year in years:
    for month in range(1, 13):
        print(f'processing data for {year}/{month}')

        input_path = f'data/raw/yellow/{year}/{month:02d}/'
        output_path = f'data/pq/yellow/{year}/{month:02d}/'

        df_yellow = spark.read.option("header", "true").schema(yellow_schema).csv(input_path)

        df_yellow.repartition(4).write.parquet(output_path)

In [29]:
# checking files (green) 
!ls -lh data/pq/green/2020/

total 48K
drwxr-xr-x 1 lourh 197609 0 May  4 20:25 01
drwxr-xr-x 1 lourh 197609 0 May  4 20:25 02
drwxr-xr-x 1 lourh 197609 0 May  4 20:25 03
drwxr-xr-x 1 lourh 197609 0 May  4 20:25 04
drwxr-xr-x 1 lourh 197609 0 May  4 20:25 05
drwxr-xr-x 1 lourh 197609 0 May  4 20:25 06
drwxr-xr-x 1 lourh 197609 0 May  4 20:25 07
drwxr-xr-x 1 lourh 197609 0 May  4 20:25 08
drwxr-xr-x 1 lourh 197609 0 May  4 20:25 09
drwxr-xr-x 1 lourh 197609 0 May  4 20:25 10
drwxr-xr-x 1 lourh 197609 0 May  4 20:25 11
drwxr-xr-x 1 lourh 197609 0 May  4 20:25 12


In [30]:
# checking files (yellow) in detail. Not just the directory
!ls -lh data/pq/yellow/2020/

total 48K
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 01
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 02
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 03
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 04
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 05
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 06
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 07
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 08
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 09
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 10
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 11
drwxr-xr-x 1 lourh 197609 0 Apr 20 19:36 12


In [51]:
# Reading parquet with pyspark
# Not sure why it did not read the data. Unless this has to be done in a new PysSpark shell as shown in notes of Manuel G.
# I suspect the partition structure is not executed correctly
df_green = spark.read.parquet('data/pq/green/*/*')

In [48]:
# Checking the Number of Files in the Output Path to verify the partitions
# Each partition will result in one file (e.g., part-00000-xxxx.parquet) in the output directory. 
# Count the number of files in the output directory: The result should be 4 if the repartitioning worked as expected.
# As we can see, it did not work

!ls -1 'data/raw/yellow/2020/01/' | grep part- | wc -l

0


In [52]:
# Not sure why it did not read the data.
# Unless this has to be done in a new PysSpark shell as shown in notes of Manuel G.
df_green.head(5)

[]

In [53]:
df_green \
    .select('lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_distance') \
    .show(5)

+--------------------+---------------------+------------+------------+-------------+
|lpep_pickup_datetime|lpep_dropoff_datetime|PULocationID|DOLocationID|trip_distance|
+--------------------+---------------------+------------+------------+-------------+
+--------------------+---------------------+------------+------------+-------------+

